### Imports and Configuration

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from pipeline_utils import make_case4_paths, binarize_mutation_inclusive

paths = make_case4_paths()

## Input files (from 0_xevaset_extraction/UHN_breast_XevaSet_extraction.R)
batch_sensitivity_file = paths["datasets"] / "uhn_breast" / "batch_sensitivity.csv"
model_sensitivity_file = paths["datasets"] / "uhn_breast" / "model_sensitivity.csv"
expdesign_file = paths["datasets"] / "uhn_breast" / "expDesign.csv"
rna_file = paths["datasets"] / "uhn_breast" / "RNASeq_matrix.csv"
cnv_file = paths["datasets"] / "uhn_breast" / "CNV_matrix.csv"
mutation_file = paths["datasets"] / "uhn_breast" / "mutation_matrix.csv"

## Output files
metadata_out = paths["proc"] / "metadata.csv"
rna_out = paths["proc"] / "rna_logtpm.csv"
cnv_out = paths["proc"] / "cnv.csv"
mutation_out = paths["proc"] / "mutation.csv"

### Drug sensitivity info extraction

In [2]:
batch_sen = pd.read_csv(batch_sensitivity_file)
model_sen = pd.read_csv(model_sensitivity_file)
exp_dsg = pd.read_csv(expdesign_file)

treatment_arm = exp_dsg[exp_dsg["arm"] == "treatment"]

# n_replicates: number of treatment-arm mice recorded per batch in expDesign.csv
n_replicates_derived = (
    treatment_arm.groupby("batch.name").size().rename("n_replicates").reset_index()
)

# mRECIST: model_sensitivity.csv records it per individual mouse, not per batch.
# Aggregate to one value per batch via majority vote across that batch's
# treatment-arm mice; ties broken by worst response (PD > SD > PR > CR).
# (Same method as 3_carboplatin's UHN/McGill notebooks.) Kept only as a
# reference/QC column -- the actual response label used for modeling is an
# angle-median-split, defined later in 2-ML.ipynb (see feature_sets.py).
MRECIST_WORST_FIRST = ["PD", "SD", "PR", "CR"]

def majority_vote_mrecist(values):
    counts = values.dropna().value_counts()
    if counts.empty:
        return None
    top_count = counts.max()
    tied = counts[counts == top_count].index
    return next(r for r in MRECIST_WORST_FIRST if r in tied)

mouse_mrecist = treatment_arm.merge(model_sen[["model.id", "mRECIST"]], on="model.id", how="left")
mrecist_derived = (
    mouse_mrecist.groupby("batch.name")["mRECIST"]
    .apply(majority_vote_mrecist)
    .rename("mRECIST")
    .reset_index()
)

# batch name contains PACLITAXEL but not resistant-derivative lines ("-"/"RES")
pax = batch_sen[
    batch_sen["batch.name"].str.contains("PACLITAXEL")
    & ~batch_sen["batch.name"].str.contains("-")
    & ~batch_sen["batch.name"].str.contains("RES")
].reset_index(drop=True)

pax["model.id"] = pax["batch.name"].str.split(".").str[0]
pax["drug.id"] = pax["batch.name"].str.split(".").str[1]

# Omics sample IDs are prefixed "S" for numeric model IDs
mask = pax["model.id"].str.match(r'^\d', na=False)
pax.loc[mask, "model.id"] = "S" + pax.loc[mask, "model.id"]

pax = pax.merge(n_replicates_derived, on="batch.name", how="left")
pax = pax.merge(mrecist_derived, on="batch.name", how="left")

n_multi_arm = (pax["n_replicates"] > 1).sum()
print(f"Paclitaxel batches with multiple treatment-arm entries (majority vote applies): {n_multi_arm}/{len(pax)}")

print(pax["mRECIST"].value_counts())

# A model's mRECIST can be missing if none of its treatment-arm mice have a
# recorded mRECIST in model_sensitivity.csv -- kept in the cohort, just flagged.
missing_mrecist = pax.loc[pax["mRECIST"].isna(), "model.id"].tolist()
if missing_mrecist:
    print(f"\nModels kept but with no recorded mRECIST: {missing_mrecist}")

print(f"\nn_replicates distribution across the {len(pax)} paclitaxel batches:")
print(pax["n_replicates"].value_counts().sort_index())

# Restrict to the columns downstream cells expect. angle is the response used
# for modeling (angle-median-split); mRECIST is kept only as a reference/QC
# column. n_replicates/drug.id are QC-only here, not persisted, so they don't
# leak into the omics feature matrices merged against this table below.
pax = pax[["model.id", "angle", "mRECIST"]].set_index("model.id")
pax.to_csv(metadata_out)

Paclitaxel batches with multiple treatment-arm entries (majority vote applies): 49/80
mRECIST
SD    29
CR    22
PD    18
PR    11
Name: count, dtype: int64

n_replicates distribution across the 80 paclitaxel batches:
n_replicates
1     31
2     19
3     17
4      4
5      1
6      1
7      1
9      1
10     2
14     1
17     1
23     1
Name: count, dtype: int64


### RNAseq data

In [3]:
# read in the RNA-seq data
rna = pd.read_csv(rna_file, index_col=0)

rna = rna.transpose()
print(f"Models in RNA-seq: {len(rna)}")
rna = np.log2(rna + 1)

# remove genes with no variance across all available RNA-seq samples
rna = rna.loc[:, rna.var() > 0]

# merge with paclitaxel response metadata
df_rna_pax = rna.merge(pax, left_index=True, right_index=True, how="inner")

# remove genes with no variance within the paclitaxel subset
response_cols = ["angle", "mRECIST"]
X = df_rna_pax.drop(columns=response_cols)
y = df_rna_pax[response_cols]
X = X.loc[:, X.var() > 0]
df_rna_pax = X.merge(y, left_index=True, right_index=True)

print(f"Models in pax list: {len(pax)}")
print(f"Models in Final DF: {df_rna_pax.shape[0]}")
print(f"RNA genes after paclitaxel subset zero-variance filter: {X.shape[1]}")

df_rna_pax.to_csv(rna_out)
print(df_rna_pax.shape)

Models in RNA-seq: 95
Models in pax list: 80
Models in Final DF: 67
RNA genes after paclitaxel subset zero-variance filter: 49348


(67, 49350)


### CNV data

In [4]:
# read in the CNV data (absolute copy number, genes x samples)
cnv = pd.read_csv(cnv_file, index_col=0)

# restrict to the paclitaxel cohort's samples up front, so the per-sample
# baseline below reflects only the models actually used downstream
cnv = cnv[cnv.columns.intersection(pax.index)]
print(f"Models in CNV before filtering: {cnv.shape[1]}")

# Per-sample baseline copy number (modal CN across the full genome, computed
# before any gene-level filtering below -- same order of operations as
# 3_carboplatin's UHN notebook, so the baseline reflects all available
# genome-wide data rather than only genes that pass the missingness filter).
# UHN CNV is not diploid at baseline for most samples; see case 3 for the
# full justification.
cnv_baseline = cnv.mode(axis=0, dropna=True).iloc[0]
print(f"Per-sample baseline copy number (modal): {cnv_baseline.value_counts().sort_index().to_dict()}")

# remove genes with high missingness across samples
missing_rate = cnv.isna().mean(axis=1)
cnv = cnv.loc[missing_rate < 0.20]
print(f"CNV genes after missingness filter: {cnv.shape[0]}")

# Ploidy-relative log2 ratio: 0 = at this sample's own baseline, positive =
# amplified relative to baseline, negative = deleted relative to baseline.
# Baseline is floored at 1 (avoids divide-by-zero for the few samples whose
# own baseline is 0, more likely low tumor purity than genuine haploid
# biology) and a +0.5 pseudocount avoids -inf for homozygous deletions (CN=0).
# This keeps the graded magnitude information a binary amp/del call would
# lose, while still correcting for each sample's own ploidy.
baseline_floor = cnv_baseline.clip(lower=1)
cnv_ratio = np.log2((cnv + 0.5).div(baseline_floor, axis=1))

# remaining missing values -> 0 (i.e. "at this sample's own baseline", the
# neutral value on this ratio scale)
cnv_ratio = cnv_ratio.fillna(0)

# transpose to samples x genes
cnv_ratio = cnv_ratio.transpose()

# remove genes with no variance across all available CNV samples
cnv_ratio = cnv_ratio.loc[:, cnv_ratio.var() > 0]
print(f"CNV genes after global zero-variance filter: {cnv_ratio.shape[1]}")

# merge with paclitaxel response metadata
df_cnv_pax = cnv_ratio.merge(pax, left_index=True, right_index=True, how="inner")

# remove genes with no variance within the paclitaxel subset
response_cols = ["angle", "mRECIST"]
X = df_cnv_pax.drop(columns=response_cols)
y = df_cnv_pax[response_cols]
X = X.loc[:, X.var() > 0]
df_cnv_pax = X.merge(y, left_index=True, right_index=True)

print(f"Models in pax list: {len(pax)}")
print(f"Models in Final DF: {df_cnv_pax.shape[0]}")
print(f"CNV genes after paclitaxel subset zero-variance filter: {X.shape[1]}")

# save processed CNV + response table
df_cnv_pax.to_csv(cnv_out)
print(df_cnv_pax.shape)

Models in CNV before filtering: 62
Per-sample baseline copy number (modal): {0.0: 3, 1.0: 7, 3.0: 40, 4.0: 4, 5.0: 5, 6.0: 2, 7.0: 1}
CNV genes after missingness filter: 10313
CNV genes after global zero-variance filter: 10313
Models in pax list: 80
Models in Final DF: 62
CNV genes after paclitaxel subset zero-variance filter: 10313


(62, 10315)


### Mutation data

In [5]:
# read in the Mutation data
mutation = pd.read_csv(mutation_file, index_col=0)

mutation = mutation.transpose()

# Binarize: 1 if any functional variant classification is present, else 0
# (same inclusive whitelist as 3_carboplatin/pipeline_utils.py)
mutation = mutation.map(binarize_mutation_inclusive)

print(f"Models in Mutation: {len(mutation)}")

# remove genes with no variance across all available mutation samples
mutation = mutation.loc[:, mutation.var() > 0]

# merge with paclitaxel response metadata
df_mutation_pax = mutation.merge(pax, left_index=True, right_index=True, how="inner")

# remove genes with no variance within the paclitaxel subset
response_cols = ["angle", "mRECIST"]
X = df_mutation_pax.drop(columns=response_cols)
y = df_mutation_pax[response_cols]
X = X.loc[:, X.var() > 0]
df_mutation_pax = X.merge(y, left_index=True, right_index=True)

print(f"Models in pax list: {len(pax)}")
print(f"Models in Final DF: {df_mutation_pax.shape[0]}")
print(f"Mutation genes after paclitaxel subset zero-variance filter: {X.shape[1]}")

df_mutation_pax.to_csv(mutation_out)
print(df_mutation_pax.shape)

Models in Mutation: 91
Models in pax list: 80
Models in Final DF: 62
Mutation genes after paclitaxel subset zero-variance filter: 13029
(62, 13031)


### Summary: sample sizes and omics overlap

In [6]:
rna_raw_n = pd.read_csv(rna_file, index_col=0, nrows=1).shape[1]
cnv_raw_n = pd.read_csv(cnv_file, index_col=0, nrows=1).shape[1]
mutation_raw_n = pd.read_csv(mutation_file, index_col=0, nrows=1).shape[1]

rna_final = pd.read_csv(rna_out, index_col=0)
cnv_final = pd.read_csv(cnv_out, index_col=0)
mutation_final = pd.read_csv(mutation_out, index_col=0)

print(f"{'':22s}{'raw total':>12s}{'∩ pax':>12s}")
print(f"{'pax (paclitaxel)':22s}{'--':>12s}{len(pax):>12d}")
print(f"{'RNA-seq':22s}{rna_raw_n:>12d}{len(rna_final):>12d}")
print(f"{'CNV':22s}{cnv_raw_n:>12d}{len(cnv_final):>12d}")
print(f"{'Mutation':22s}{mutation_raw_n:>12d}{len(mutation_final):>12d}")

cnv_mut_same = set(cnv_final.index) == set(mutation_final.index)
print(f"\nCNV and Mutation cover the identical model set: {cnv_mut_same}")

rna_only = sorted(set(rna_final.index) - set(cnv_final.index))
three_way = set(rna_final.index) & set(cnv_final.index) & set(mutation_final.index)
print(f"RNA-only models (no CNV/Mutation data): {rna_only}")
print(f"\nTrue 3-way overlap (RNA & CNV & Mutation): {len(three_way)} models")
print("-> this is the cohort available for early/late fusion in 2-ML.ipynb")

                         raw total       ∩ pax
pax (paclitaxel)                --          80
RNA-seq                         95          67
CNV                             91          62
Mutation                        91          62

CNV and Mutation cover the identical model set: True
RNA-only models (no CNV/Mutation data): ['NOTCH01', 'NOTCH06', 'NOTCH10', 'NOTCH18', 'NOTCHB02']

True 3-way overlap (RNA & CNV & Mutation): 62 models
-> this is the cohort available for early/late fusion in 2-ML.ipynb
